<a href="https://colab.research.google.com/github/Skulfre/Machine-Learning/blob/main/transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

In [ ]:
# Continue with regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
    from going_modular.going_modular import data_setup, engine, train
except:
    # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine

In [ ]:
device = "cuda" if torch.cuda.is_available() else 'cpu'
device

In [ ]:
!nvidia-smi

In [ ]:
import os
import zipfile
from pathlib import Path

# Download data...
import os
import zipfile

from pathlib import Path

import requests

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder doesn't exist, download it and prepare it...
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

    # Download pizza, steak, sushi data
    with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
        print("Downloading pizza, steak, sushi data...")
        f.write(request.content)

    # Unzip pizza, steak, sushi data
    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
        print("Unzipping pizza, steak, sushi data...")
        zip_ref.extractall(image_path)

    # Remove .zip file
    os.remove(data_path / "pizza_steak_sushi.zip")

In [ ]:
# Setup Dirs
train_dir = image_path / "train"
test_dir = image_path / "test"

In [ ]:
# Creating a manual transform for Tranform Learning Model
import torchvision
from torchvision import transforms

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

"""
Ovo su mean i std ImageNet dataseta — ogromnog dataseta od ~1.2 miliona slika na kojima su trenirani modeli poput ResNet, VGG, EfficientNet itd.
Neko je bukvalno uzeo sve te slike, izračunao prosječnu vrijednost i standardnu devijaciju po svakom kanalu (R, G, B) — i dobio te brojeve.
"""

manual_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=manual_transform,
                                                                               batch_size=32)

train_dataloader, test_dataloader, class_names

In [ ]:
# Creating a transform for torchvision.models (auto creation)

from torchvision import models
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT


"""
it's important that your custom data going into the model is prepared in
the same way as the original training data that went into the model.

EfficientNet_B0_Weights is the model architecture weights we'd like to use (there are many different model architecture options in torchvision.models).
DEFAULT means the best available weights (the best performance in ImageNet).
"""

# Get a set of pretrained model weights
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT # .DEFAULT = best available weights from pretraining on ImageNet
weights

"""
And now to access the transforms associated with our weights, we can use the transforms() method.
This is essentially saying "get the data transforms that were used to train the EfficientNet_B0_Weights on ImageNet".
"""

auto_transforms = weights.transforms()
auto_transforms

In [ ]:
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=auto_transforms,
                                                                               batch_size=32)
train_dataloader, test_dataloader, class_names

In [ ]:
# Old method of creating a pretrained model
# model = torchvision.models.efficientnet_b0(pretrained=True)

# New method of creating a pretrained model
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model = torchvision.models.efficientnet_b0(weights=weights).to(device)
model

In [ ]:
model.features

In [ ]:
model.avgpool

In [ ]:
model.classifier

In [ ]:
# Getting a summary of our model using torchinfo
!pip install -q torchinfo

In [ ]:
from torchinfo import summary
summary(model=model,
        input_size=(1, 3, 224, 224),
        # Example of [batch_size, color_channels, height, width]
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"],
)

In [ ]:
# Freezing the base model and changing the output layer to suit our needs
for params in model.features.parameters():
  params.requires_grad = False # Turning off gradient tracking so it wont update params

summary(model=model,
        input_size=(1, 3, 224, 224),
        # Example of [batch_size, color_channels, height, width]
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"],
)

In [ ]:
model.classifier

In [ ]:
from torch import nn
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, out_features=len(class_names))
).to(device)

In [ ]:
model.classifier

In [ ]:
summary(model=model,
        input_size=(1, 3, 224, 224),
        # Example of [batch_size, color_channels, height, width]
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"],
)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

In [ ]:
from going_modular.going_modular import engine

results = engine.train(model=model,
                       train_dataloader = train_dataloader,
                       test_dataloader = test_dataloader,
                       optimizer = optimizer,
                       loss_fn = loss_fn,
                       epochs = 7,
                       device = device)

In [ ]:
try:
  from helper_functions import plot_loss_curves
  print("Failed...")
except:
  print(f"[INFO] Couldnt find helper_functions.py, downloading")
  with open('helper_functions.py', 'wb') as f:
    import requests
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/helper_functions.py")
    f.write(request.content)
from helper_functions import plot_loss_curves

In [ ]:
# Plot the Loss Curves
plot_loss_curves(results)

In [ ]:
from typing import List, Tuple
from PIL import Image

def pred_and_plot_image(model: torch.nn.Module,
                        image_path: str,
                        class_names: List[str],
                        image_size: Tuple[int, int] = (224, 224), # Taking width and height not RGB
                        transform: torchvision.transforms = None,
                        device: torch.device = device):

  img = Image.open(image_path)

  # Create a transform is one doesn't exist
  if transform != None:
    image_transform = transform
  else:
    image_transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(), # Automatically adding an RGB
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std = [0.229, 0.224, 0.225])
    ])

  model.to(device)

  model.eval()
  with torch.inference_mode():
    transformed_image = image_transform(img).unsqueeze(dim=0).to(device)
    logits = model(transformed_image)
  pred_probs = torch.softmax(logits, dim=1)
  pred_labels = torch.argmax(pred_probs, dim=1)

  plt.figure(figsize=(10, 7))
  plt.imshow(img)
  plt.title(f"Prediction: {class_names[pred_labels]} |  Probs: {pred_labels.item()}")
  plt.axis(False)

In [ ]:
pred_and_plot_image(model=model,
                    image_path="data/pizza_steak_sushi/train/pizza/1008844.jpg",
                    class_names=class_names)

In [ ]:
import random
all_paths = list(test_dir.glob("*/*.jpg"))
random_samples = random.sample(all_paths, k=10)
random_samples

In [ ]:
for x in random_samples:
  pred_and_plot_image(model=model,
                    image_path=x,
                    class_names=class_names)

In [ ]:
cat going_modular/going_modular/train.py

In [ ]:
all_preds = []
model.eval()
with torch.inference_mode():
  for image, truth_label in test_dataloader:
    image, truth_label = image.to(device), truth_label.to(device)

    logits = model(image)
    pred_prob = torch.softmax(logits, dim=1)
    pred_label = torch.argmax(pred_prob, dim=1)
    # print(pred_prob.max(dim=1).values)

    for i in range(len(image)):
      if truth_label[i] != pred_label[i]: # svaku sliku a ne cijeli batch
        all_preds.append({
            "image": image[i].cpu(),
            "truth_label": truth_label[i].item(),
            "pred_probs": pred_prob[i].max().item(),
            "pred_labels": pred_label[i].item()
        })


def return_pred_prob(lista):
  return lista['pred_probs']

all_preds.sort(key=return_pred_prob, reverse=True)

all_preds[:5]


In [ ]:
most_wrong = all_preds[:5]
most_wrong[0]['image']

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


for x in most_wrong:
  plt.figure(figsize=(10, 7))
  plt.axis(False)
  image = x['image']
  plt.title(f"Truth Label: {class_names[x['truth_label']]} |  Predicted Label: {class_names[x['pred_labels']]}")
  plt.imshow(image.permute(1, 2, 0))



In [ ]:
from pathlib import Path
import requests
import zipfile

data_path = Path('data/')
image_path = data_path / "pizza_steak_sushi_20_percent"

if image_path.is_dir():
  print(f"File path {image_path} already exist")
else:
  print(f"Creating a file path: {image_path} inside of {data_path}...")
  image_path.mkdir(parents=True, exist_ok=True)

  with open(data_path / "pizza_steak_sushi_20_percent.zip", 'wb') as f:
     request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip")
     print(f"Downloading data...")
     f.write(request.content)

  with zipfile.ZipFile(data_path / "pizza_steak_sushi_20_percent.zip", 'r') as zip_ref:
    print("Unzipping files...")
    zip_ref.extractall(image_path)

In [ ]:
train_dir_20 = image_path / "train"
test_dir_20 = image_path / "test"
train_dir_20, test_dir_20

In [ ]:
weights_2 = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model_2 = torchvision.models.efficientnet_b0(weights=weights_2).to(device)
auto_transform_2 = weights_2.transforms()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_2.parameters(), lr=0.001)

In [ ]:
train_dataloader_20, test_dataloader_20, class_names_20 = data_setup.create_dataloaders(train_dir=train_dir_20,
                                                                                        test_dir=test_dir_20,
                                                                                        transform=auto_transforms,
                                                                                        batch_size=32)

In [ ]:
engine.train(model=model_2,
             train_dataloader=train_dataloader_20,
             test_dataloader=test_dataloader_20,
             optimizer=optimizer,
             loss_fn=loss_fn,
             epochs=10,
             device=device)

In [ ]:
weights_3 = torchvision.models.ResNet18_Weights.DEFAULT
model_3 = torchvision.models.resnet18(weights=weights_3).to(device)

In [ ]:
print(model_3)

In [ ]:
model_3.fc

In [ ]:
model_3.fc = nn.Linear(
    in_features=512,
    out_features=3
)

In [ ]:
auto_transform_3 = weights_3.transforms()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_3.parameters(), lr=0.001)

In [ ]:
engine.train(model=model_3,
             train_dataloader=train_dataloader_20,
             test_dataloader=test_dataloader_20,
             optimizer=optimizer,
             loss_fn=loss_fn,
             epochs=10,
             device=device)

In [ ]:
weights_4 = torchvision.models.EfficientNet_B3_Weights.DEFAULT
model_4 = torchvision.models.efficientnet_b3(weights=weights_4).to(device)

In [ ]:
model_4.classifier

In [ ]:
model_4.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features=1536, out_features=3)
)

In [ ]:
for params in model_4.features.parameters():
  params.requires_grad = False

In [ ]:
auto_transform_4 = weights_4.transforms()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_4.parameters(), lr=0.001)

In [ ]:
engine.train(model=model_4,
             train_dataloader=train_dataloader_20,
             test_dataloader=test_dataloader_20,
             optimizer=optimizer,
             loss_fn=loss_fn,
             epochs=5,
             device=device)